In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

In [4]:
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
processed_data_dir = project_root / "data" / "processed"
raw_data_dir = project_root / "data" / "raw"
oof_dir = project_root / "artifacts" / "oof"
test_predictions_dir = project_root / "artifacts" / "test_predictions"
metrics_dir = project_root / "artifacts" / "metrics"

train_path = processed_data_dir / "train_with_folds.parquet"
test_path = processed_data_dir / "test_clean.parquet"
sample_submission_path = raw_data_dir / "sample_submission.csv"

train_df = pd.read_parquet(train_path)
test_df = pd.read_parquet(test_path)
sample_submission_df = pd.read_csv(sample_submission_path)

train_df.shape, test_df.shape, sample_submission_df.shape

((699635, 25), (299844, 23), (299844, 2))

In [5]:
target = train_df["satisfaction"]
fold_numbers = train_df["fold"]
train_ids = train_df["id"]
test_ids = test_df["id"]

features = train_df.drop(columns=["id", "satisfaction", "fold"])
test_features = test_df.drop(columns=["id"])

class_values = sorted(features["Class"].unique())
features["Class"] = pd.Categorical(features["Class"], categories=class_values)
test_features["Class"] = pd.Categorical(test_features["Class"], categories=class_values)

assert list(features.columns) == list(test_features.columns)

features.shape, test_features.shape, features["Class"].dtype

((699635, 22),
 (299844, 22),
 CategoricalDtype(categories=['Business', 'Eco', 'Eco Plus'], ordered=False, categories_dtype=str))

In [6]:
fold = 0

train_mask = fold_numbers != fold
valid_mask = fold_numbers == fold

x_train = features.loc[train_mask]
x_valid = features.loc[valid_mask]

y_train = target.loc[train_mask]
y_valid = target.loc[valid_mask]

x_train.shape, x_valid.shape, y_train.shape, y_valid.shape

((559708, 22), (139927, 22), (559708,), (139927,))

In [7]:
model = XGBClassifier(
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    enable_categorical=True,
    early_stopping_rounds=100,
    random_state=42,
    n_jobs=-1,
)

In [ ]:
model.fit(
    x_train,
    y_train,
    eval_set = [(x_valid, y_valid)],
    verbose = 100
)

valid_predictions = model.predict_proba(x_valid)[:, 1]

auc = roc_auc_score(y_valid, valid_predictions)

print("Best iteration:", model.best_iteration)
print("Validation ROC AUC:", auc)